# Phase 3a: PubMed Medical Summarization

AutoDL H800 80GB — 医疗摘要实验（跨领域验证）

| # | 实验 | 方法 | 数据集 | 模型 |
|---|------|------|--------|------|
| E15 | LoRA PubMed Qwen    | LoRA (r=16) | PubMed Summ | Qwen2.5-1.5B |
| E16 | LoRA PubMed Llama   | LoRA (r=16) | PubMed Summ | Llama-3.2-1B |
| E17 | Full FT PubMed Qwen | Full FT     | PubMed Summ | Qwen2.5-1.5B |
| E18 | Full FT PubMed Llama| Full FT     | PubMed Summ | Llama-3.2-1B |
| E19 | Random PubMed Qwen  | LoRA (shuffled labels) | PubMed Summ | Qwen2.5-1.5B |
| E20 | Random PubMed Llama | LoRA (shuffled labels) | PubMed Summ | Llama-3.2-1B |

## 0. 环境准备

In [ ]:
import os, sys
os.chdir('/root/MLP')
os.environ['HF_HOME'] = '/root/autodl-tmp/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/root/autodl-tmp/hf_cache'
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['WANDB_MODE'] = 'offline'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.makedirs('/root/autodl-tmp/hf_cache', exist_ok=True)
os.makedirs('/root/autodl-tmp/outputs', exist_ok=True)
os.makedirs('/root/autodl-tmp/logs', exist_ok=True)

# 软链到数据盘
for target, link in [('/root/autodl-tmp/outputs', '/root/MLP/outputs'),
                      ('/root/autodl-tmp/logs',    '/root/MLP/logs'),
                      ('/root/autodl-tmp/data',    '/root/MLP/data')]:
    if not os.path.islink(link):
        if os.path.isdir(link):
            print(f'  ⚠ {link} is a real dir, skipping')
        else:
            os.symlink(target, link)
            print(f'  ✓ created: {link} -> {target}')
    else:
        print(f'  ✓ symlink exists: {link} -> {os.readlink(link)}')

!pwd && ls
print(f'Python: {sys.executable}')
print(f'HF_HOME: {os.environ["HF_HOME"]}')
print(f'HF_ENDPOINT: {os.environ["HF_ENDPOINT"]}')
print(f'WANDB_MODE: {os.environ["WANDB_MODE"]}')

In [ ]:
import sys
!{sys.executable} -m pip install peft accelerate trl bitsandbytes wandb rouge-score bert-score scikit-learn sentencepiece huggingface_hub datasets
print('\n✅ 安装完成！如果是第一次装，请重启内核：Kernel → Restart Kernel')

In [ ]:
import torch, peft, trl
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

In [ ]:
# HuggingFace 登录 (Llama 需要)
from huggingface_hub import login
login()
print('HuggingFace 登录成功')

## 1. 数据下载与格式化

从 HuggingFace 下载 `ccdv/pubmed-summarization`，采样 20k 训练集，格式化为 SFT JSONL。

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/data/pubmed/pubmed_formatting.py --train_samples 20000 --output_dir data/pubmed
print('数据格式化完成')

In [ ]:
# 验证数据文件
import os, json
files = [
    'data/pubmed/train_sft.jsonl',
    'data/pubmed/val_sft.jsonl',
    'data/pubmed/test_sft.jsonl',
]
for f in files:
    if os.path.exists(f):
        with open(f) as fh:
            n = sum(1 for _ in fh)
        size = os.path.getsize(f) // (1024 * 1024)
        print(f'✓ {f}: {n:,} records, {size} MB')
    else:
        print(f'✗ {f}: 缺失！')

# 看第一条数据
with open('data/pubmed/train_sft.jsonl') as f:
    sample = json.loads(f.readline())
print(f'\n--- Sample ---')
print(f'Input length: {len(sample["input"])} chars')
print(f'Output length: {len(sample["output"])} chars')
print(f'Output preview: {sample["output"][:200]}...')

---
## 2. LoRA 实验

### E15: LoRA — PubMed × Qwen2.5-1.5B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
os.makedirs('logs', exist_ok=True)
!{sys.executable} -u src/train/train.py --config configs/lora_pubmed_qwen.yaml 2>&1 | tee logs/lora_pubmed_qwen.log
print('训练完成')

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.makedirs('results/pubmed', exist_ok=True)
!{sys.executable} -u src/evaluate/inference.py --config configs/lora_pubmed_qwen.yaml --split test --batch_size 8
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/lora_pubmed_qwen/predictions_test.jsonl --output results/pubmed/lora_qwen_test.json
print('评估完成')
!cat results/pubmed/lora_qwen_test.json

### E16: LoRA — PubMed × Llama-3.2-1B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/lora_pubmed_llama.yaml 2>&1 | tee logs/lora_pubmed_llama.log
print('训练完成')

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/lora_pubmed_llama.yaml --split test --batch_size 8
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/lora_pubmed_llama/predictions_test.jsonl --output results/pubmed/lora_llama_test.json
print('评估完成')
!cat results/pubmed/lora_llama_test.json

---
## 3. Full Fine-Tuning 实验

### E17: Full FT — PubMed × Qwen2.5-1.5B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/full_pubmed_qwen.yaml 2>&1 | tee logs/full_pubmed_qwen.log
print('训练完成')

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/full_pubmed_qwen.yaml --split test --batch_size 8
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/full_pubmed_qwen/predictions_test.jsonl --output results/pubmed/full_qwen_test.json
print('评估完成')
!cat results/pubmed/full_qwen_test.json

### E18: Full FT — PubMed × Llama-3.2-1B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/full_pubmed_llama.yaml 2>&1 | tee logs/full_pubmed_llama.log
print('训练完成')

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/full_pubmed_llama.yaml --split test --batch_size 8
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/full_pubmed_llama/predictions_test.jsonl --output results/pubmed/full_llama_test.json
print('评估完成')
!cat results/pubmed/full_llama_test.json

---
## 4. Random Label Baseline

将训练集的 output 打乱（input 不变），验证模型是否真正学到了任务知识。

### 4.0 生成 Random Label 数据

In [ ]:
import json, random
from pathlib import Path

random.seed(42)

def shuffle_labels(input_path, output_path):
    records = []
    with open(input_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    outputs = [r['output'] for r in records]
    random.shuffle(outputs)
    for rec, new_out in zip(records, outputs):
        rec['output'] = new_out
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    print(f'  ✓ {output_path}: {len(records):,} records (labels shuffled)')

print('=== Generating Random Label PubMed Datasets ===')
shuffle_labels('data/pubmed/train_sft.jsonl', 'data/pubmed/train_sft_random.jsonl')
shuffle_labels('data/pubmed/val_sft.jsonl',   'data/pubmed/val_sft_random.jsonl')
print('\nDone. Test file is NOT shuffled (evaluate on real data).')

In [ ]:
# 验证 mismatch rate
import json

def mismatch_rate(orig_path, random_path):
    with open(orig_path) as f1, open(random_path) as f2:
        orig = [json.loads(l)['output'] for l in f1 if l.strip()]
        rand = [json.loads(l)['output'] for l in f2 if l.strip()]
    return sum(a != b for a, b in zip(orig, rand)) / len(orig)

print('Mismatch rates (should be ~1.0):')
print(f'  PubMed train: {mismatch_rate("data/pubmed/train_sft.jsonl", "data/pubmed/train_sft_random.jsonl"):.4f}')
print(f'  PubMed val:   {mismatch_rate("data/pubmed/val_sft.jsonl", "data/pubmed/val_sft_random.jsonl"):.4f}')

### E19: Random Label LoRA — PubMed × Qwen2.5-1.5B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/random_pubmed_qwen.yaml 2>&1 | tee logs/random_pubmed_qwen.log
print('训练完成')

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/random_pubmed_qwen.yaml --split test --batch_size 8
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/random_pubmed_qwen/predictions_test.jsonl --output results/pubmed/random_qwen_test.json
print('评估完成')
!cat results/pubmed/random_qwen_test.json

### E20: Random Label LoRA — PubMed × Llama-3.2-1B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/random_pubmed_llama.yaml 2>&1 | tee logs/random_pubmed_llama.log
print('训练完成')

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/random_pubmed_llama.yaml --split test --batch_size 8
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/random_pubmed_llama/predictions_test.jsonl --output results/pubmed/random_llama_test.json
print('评估完成')
!cat results/pubmed/random_llama_test.json

---
## 5. 汇总所有 PubMed 结果

In [ ]:
import json, os

results = [
    ('LoRA  × Qwen',        'results/pubmed/lora_qwen_test.json'),
    ('LoRA  × Llama',       'results/pubmed/lora_llama_test.json'),
    ('Full FT × Qwen',     'results/pubmed/full_qwen_test.json'),
    ('Full FT × Llama',    'results/pubmed/full_llama_test.json'),
    ('Random × Qwen',      'results/pubmed/random_qwen_test.json'),
    ('Random × Llama',     'results/pubmed/random_llama_test.json'),
]

print(f'{"实验":<25} {"ROUGE-1":>10} {"ROUGE-2":>10} {"ROUGE-L":>10} {"BERTScore":>10}')
print('-' * 70)
for name, path in results:
    if not os.path.exists(path):
        print(f'{name:<25} {"未完成":>10}')
        continue
    with open(path) as f:
        d = json.load(f)
    r1 = d['rouge1']['mean'] if isinstance(d['rouge1'], dict) else d['rouge1']
    r2 = d['rouge2']['mean'] if isinstance(d['rouge2'], dict) else d['rouge2']
    rl = d['rougeL']['mean'] if isinstance(d['rougeL'], dict) else d['rougeL']
    bs = d['bertscore_f1']['mean'] if isinstance(d['bertscore_f1'], dict) else d['bertscore_f1']
    print(f'{name:<25} {r1:>10.4f} {r2:>10.4f} {rl:>10.4f} {bs:>10.4f}')